## huggingface api token

In [ ]:
# for read token
from google.colab import userdata
read_key = userdata.get('token_read_hf')
# for write token
from google.colab import userdata
write_key = userdata.get('write_token_hf')

In [ ]:
read_key

In [ ]:
write_key

# libaries install

In [ ]:
!pip install --upgrade -q huggingface_hub

# Hub libaries cover
- here we make repo create and delete and upload file and delte file etc

# Part 1: Login & Authentication

First lets understand in how many ways you can login to HF
- CLI
- Via Api
- Via Notebook

### Frist is CLI

In [ ]:
#This command tells Git to save your credentials (username + password or token) permanently in a plain text file on your computer,
#so you don’t have to enter them again when pushing/pulling from a remote repo.

# more guidness on this link
# https://huggingface.co/docs/huggingface_hub/index
# and this link have both command cli and code and you can any use

Note : here login with write token not read because we will make repo on huggingface

In [ ]:
!hf auth --help

In [ ]:
!hf auth login

In [ ]:
!hf auth whoami

- for this one you will have to use token with write permission

In [ ]:
!hf repos create iabubakar/livehfcourse --repo-type model

In [ ]:
!hf repos create iabubakar/livehfdatacourse --repo-type dataset

In [ ]:
!hf repos delete iabubakar/livehfdatacourse --repo-type dataset

# Seond Via API

In [ ]:
import os
from huggingface_hub import HfApi

In [ ]:
api = HfApi(token=write_key)

In [ ]:
from huggingface_hub import create_repo
api.create_repo(
    repo_id="iabubakar/sunnyrepo",  # full path
    repo_type="model",                         # or just skip (default is model)
    private=False                              # set True for private repo
)

In [ ]:
from huggingface_hub import create_repo
api.create_repo("iabubakar/test-dataset", repo_type="dataset",private=False)

In [ ]:
api.delete_repo(repo_id="iabubakar/test-dataset", repo_type="dataset")

In [ ]:
api.delete_repo(repo_id="iabubakar/test-model", repo_type="model")

### Third Via login

In [ ]:
from huggingface_hub import login

login(read_key)

In [ ]:
from huggingface_hub import login

login(write_key)

### Notebook login

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Dataset libary

- more details here : https://huggingface.co/docs/datasets/index

Install datasets library

In [ ]:
!pip install datasets fsspec #Filesystem Specification

### Load the Dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/imdb")

In [ ]:
print(dataset)

In [ ]:
dataset['train'][0]

In [ ]:
print(dataset["train"].features)
print(dataset["train"].column_names)
print(dataset["train"].features["label"].names)
print(dataset["train"].num_rows)
print(dataset["train"].num_columns)
print(dataset["train"].shape)

In [ ]:
dataset['train'][:3]

In [ ]:
# manaully
# shuffled = dataset["train"].shuffle(seed=42)
# shuffled.select(range(100))

# in one line
# Random 5,000 samples from train split
subset_train = dataset["train"].shuffle(seed=42).select(range(5000))

In [ ]:
#note:
# you can perfrom preprocessing on datase

streaming=True
⚠️ Very important!

Normally, Hugging Face datasets are downloaded completely to disk before being used (especially for large datasets, this can be slow/heavy).

When streaming=True:

The dataset is not downloaded in full

Data is streamed sample-by-sample on-the-fly

Useful for huge datasets that can’t fit into memory (like c4, the_pile, openwebtext)

You get a generator-style iterable dataset, not an in-memory list

In [ ]:
big_data = load_dataset("openwebtext", streaming=True)

In [ ]:
big_data

| Part                  | Meaning                                                 |
| --------------------- | ------------------------------------------------------- |
| `IterableDatasetDict` | Dictionary-like object with splits (`train`, etc.)      |
| `train`               | Only split provided in `openwebtext`                    |
| `features: ['text']`  | Each example is a dict: `{"text": "...large block..."}` |
| `num_shards: 21`      | Internally it's chunked into 21 streamable parts        |


Dataset: openwebtext

Size: ~8.13 million documents

Total tokens: ~40+ billion tokens

Size on disk: ~38 GB

### Custom dataset

In [ ]:
from datasets import Dataset
import pandas as pd

data = {
    "text": [
        "I love this product! Works great ",
        "Terrible experience, I want a refund.",
        "Fast delivery and excellent packaging.",
        "Worst app I’ve ever used. Crashed in 1 minute.",
        "Super helpful support team, thanks a lot!",
        "Nothing worked. Waste of time."
    ],
    "label": [1, 0, 1, 0, 1, 0]  # 1 = Positive, 0 = Negative
}

In [ ]:
data

In [ ]:
df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df)

In [ ]:
dataset

In [ ]:
dataset = dataset.class_encode_column("label")  # optional: make label a ClassLabel

In [ ]:
dataset

In [ ]:
# push data
dataset.push_to_hub("iabubakar/mycustomerdata")

# transformer library

# tokenization

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
def tokenize(example):
    return tokenizer(example, truncation=True, padding="max_length")

In [ ]:
dataset['text'][0]

In [ ]:
tokenize(dataset["text"][0])

In [ ]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")
encoded_dataset = dataset.map(tokenize, batched=True)


In [ ]:
encoded_dataset

In [ ]:
encoded_dataset['input_ids'][0]

In [ ]:
encoded_dataset['token_type_ids'][0]

In [ ]:
# some e.g tokenize
tokens=tokenizer.tokenize("Hello HuggingFace!")
tokens

In [ ]:
tokens = tokenizer.tokenize("Hello, how are you?")
tokens

In [ ]:
inputs = tokenizer("Hello, how are you?", return_tensors="pt")
inputs

1. input_ids:

These are the IDs of the tokens from the model’s vocabulary.

The sentence "Hello, how are you?" gets tokenized as:

[CLS] Hello , how are you ? [SEP]

Which corresponds to:

[101, 7592, 1010, 2129, 2024, 2017, 1029, 102]

2. token_type_ids:

These are used in tasks like Next Sentence Prediction.

0 means it's part of the first sentence

If there was a second sentence, it would have 1s

Since your input is just a single sentence, all values are 0.

3. attention_mask:

Tells the model which tokens are real and which are padding.

1 means the token is real (not padding)

0 means it’s padding (none here)

What does return_tensors="pt" do?

It returns everything as PyTorch tensors (pt = PyTorch).

If you use return_tensors="tf", you'll get TensorFlow tensors.

### How to tokenize large dataset faster

In [ ]:
import time
text = ["This is a sample sentence for tokenization."] * 100000

In [ ]:
# Fast
start = time.time()
tok_fast = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=True)
tok_fast(text, padding=True, truncation=True)
print("Fast Time:", time.time() - start)

In [ ]:
# Slow
start = time.time()
tok_slow = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
tok_slow(text, padding=True, truncation=True)
print("Slow Time:", time.time() - start)

| Feature         | Fast Tokenizer (`use_fast=True`)       | Slow Tokenizer (`use_fast=False`) |
| --------------- | -------------------------------------- | --------------------------------- |
| Backend         | Built in **Rust** (super fast)       | Built in **Python** (slower)   |
| Tokenizer Class | `BertTokenizerFast`                    | `BertTokenizer`                   |
| Speed           | Very fast                           | Much slower                    |
| Library Used    | `tokenizers` (Rust)                    | `transformers` (pure Python)      |
| Memory usage    | Slightly higher (due to Rust overhead) | Lower                             |


## Do you wanna create your own tokenizer?

### Here You're training your own custom tokenizer from scratch using the Byte Pair Encoding (BPE) algorithm, with Hugging Face’s fast tokenizers library.

Below code creates a custom tokenizer from your own small dataset using the BPE algorithm, with:

Whitespace splitting,

100-token vocabulary,

Trained from scratch.


Byte Pair Encoding (BPE) is a subword tokenization algorithm that helps break down words into smaller pieces (subwords), making it especially useful for handling rare words and unknown words in NLP tasks.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
corpus = ["I love transformers!", "Tokenizers are amazing.", "Build your own.\n"]

In [ ]:
corpus

In [ ]:
tokenizer = Tokenizer(models.BPE())

In [ ]:
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

In [ ]:
trainer = trainers.BpeTrainer(vocab_size=100)

In [ ]:
tokenizer.train_from_iterator(corpus, trainer)

In [ ]:
# Save
tokenizer.save("custom-tokenizer.json")

In [ ]:
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="/content/custom-tokenizer.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

In [ ]:
hf_tokenizer.save_pretrained("my_tokenizer_hf")

- some more prectical tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DistilBertTokenizer, DistilBertForSequenceClassification

In [ ]:
model_name2 = "nlptown/bert-base-multilingual-uncased-sentiment"
mymodel2 = AutoModelForSequenceClassification.from_pretrained(model_name2)
mytokenizer2 = AutoTokenizer.from_pretrained(model_name2)

classifier = pipeline("sentiment-analysis", model = mymodel2 , tokenizer = mytokenizer2)
res = classifier("I was so not happy with the Barbie Movie")
print(res)

In [ ]:
from transformers import AutoTokenizer

# Load a pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Example text
text = "I was so not happy with the Barbie Movie"

# Tokenize the text
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

# Convert tokens to input IDs
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print("Input IDs:", input_ids)

# Encode the text (tokenization + converting to input IDs)
encoded_input = tokenizer(text)
print("Encoded Input:", encoded_input)

# Decode the text
decoded_output = tokenizer.decode(input_ids)
print("Decode Output: ", decoded_output)

**token_type_ids**<br>
These IDs are used to distinguish between different sequences in tasks that involve multiple sentences, such as question-answering and sentence-pair classification. BERT uses this mechanism to understand which tokens belong to which segment. For single-sequence tasks like sentiment analysis, token_type_ids are all zeros.

**attention_mask** <br>
The attention mask is used to differentiate between actual tokens and padding tokens (if any). It helps the model focus on non-padding tokens and ignore padding tokens. A value of 1 indicates that the token should be attended to, while a value of 0 indicates padding.

**Why Padding Tokens Are Used**<br>
Uniform Sequence Length: Deep learning models typically process input data in batches. To efficiently process these batches, all sequences in a batch must have the same length. Padding tokens ensure this by extending shorter sequences to match the length of the longest sequence in the batch.
Efficient Computation: Fixed-length sequences allow for more efficient use of hardware resources, as the model can process all sequences in parallel without needing to handle variable-length sequences individually.



### Now load the model

In [ ]:
from transformers import AutoModel
model = AutoModel.from_pretrained("bert-base-uncased")

## sentence transformer library
###### sentence transformer library: https://huggingface.co/sentence-transformers

In [ ]:
sentence1 = "Hello, how are you?"
sentence2 = "Hi, how do you do?"

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Step 1: Encode both sentences into embeddings
embedding1 = model.encode(sentence1, convert_to_tensor=True)

In [ ]:
len(embedding1)

In [ ]:
embedding2 = model.encode(sentence2, convert_to_tensor=True)

In [ ]:
len(embedding2)

In [ ]:
from torch.nn.functional import cosine_similarity
similarity = cosine_similarity(embedding1, embedding2, dim=0)
print(f"Similarity: {similarity.item():.4f}")

## Example of AutoModelClass with Different heads

| Task                        | Class Name                                          | Description                                          |
| --------------------------- | --------------------------------------------------- | ---------------------------------------------------- |
| Base Model               | `AutoModel`                                         | Just embeddings (no head), for feature extraction    |
| Masked LM                | `AutoModelForMaskedLM`                              | For BERT-style `[MASK]` prediction                   |
| Sequence Classification | `AutoModelForSequenceClassification`                | For sentence-level tasks like sentiment, spam        |
| Token Classification     | `AutoModelForTokenClassification`                   | For NER, POS tagging (token-level labels)            |
| Multiple Choice          | `AutoModelForMultipleChoice`                        | For MCQ tasks (e.g., SWAG dataset)                   |
| Next Sentence Prediction | `AutoModelForNextSentencePrediction`                | For NSP (mainly BERT pretraining)                    |
| Causal LM (Text Gen)     | `AutoModelForCausalLM`                              | For GPT-style models (left-to-right text generation) |
| Seq2Seq LM               | `AutoModelForSeq2SeqLM`                             | For translation, summarization (T5, BART)            |
| Question Answering       | `AutoModelForQuestionAnswering`                     | For extractive QA (SQuAD-style)                      |
| Vision Tasks            | `AutoModelForImageClassification`, etc.             | For image classification (e.g., ViT)                 |
| Audio Tasks              | `AutoModelForAudioClassification`, etc.             | For Wav2Vec, Whisper, etc.                           |
| Conditional Generation   | `AutoModelForConditionalGeneration`                 | For T5/BART-style seq2seq tasks                      |
| Zero-shot Tasks          | `AutoModelForZeroShotClassification` (via pipeline) | For inference without fine-tuning                    |
| Contrastive Learning     | `AutoModelForContrastiveLearning`                   | For embeddings comparison tasks                      |


# pipeline

## Inbuilt Pipeline method

| Parameter           | Type                        | Description / Use Case                                                               |
| ------------------- | --------------------------- | ------------------------------------------------------------------------------------ |
| `task`              | `str`                       | Task name like `"text-classification"`, `"text-generation"`, `"summarization"`, etc. |
| `model`             | `str` or `PreTrainedModel`  | Model name (e.g., `"bert-base-uncased"`) or loaded model object                      |
| `tokenizer`         | `str` or Tokenizer Object   | Optional – auto-loaded from model if not given                                       |
| `config`            | `str` or `PretrainedConfig` | Optional – custom configuration (e.g., num labels, dropout, etc.)                    |
| `framework`         | `str` (`"pt"` or `"tf"`)    | Force PyTorch or TensorFlow (auto-detected if not set)                               |
| `device`            | `int` or `str`              | `0` = GPU, `-1` = CPU, `"cuda"` or `"cpu"`                                           |
| `revision`          | `str`                       | Git branch/tag/commit from Hugging Face Hub (e.g., `"main"`, `"v1.0"`)               |
| `use_fast`          | `bool`                      | Whether to use fast tokenizer (defaults to `True`)                                   |
| `token`             | `str` or `bool`             | Hugging Face token (needed for private models or rate limits)                        |
| `device_map`        | `Any`                       | For multi-GPU / model parallelism                                                    |
| `torch_dtype`       | `torch.dtype`               | Force precision: e.g., `torch.float16` for faster inference                          |
| `trust_remote_code` | `bool`                      | Allow loading custom `model.py` logic from HF repo (for advanced/custom models)      |
| `model_kwargs`      | `dict`                      | Extra arguments to pass to model (e.g., `temperature`, `top_p`, `max_length`, etc.)  |
| `feature_extractor` | `str` or Object             | Used for older vision/audio models (now replaced by `image_processor`)               |
| `image_processor`   | `BaseImageProcessor`        | For image tasks (used in ViT, SAM, etc.)                                             |
| `processor`         | `ProcessorMixin`            | For multi-modal models (e.g., Whisper, CLIP, LayoutLMv3)                             |
| `pipeline_class`    | `Any`                       | Custom pipeline class if you’re extending or modifying the base pipeline logic       |
| `**kwargs`          | `Any`                       | Additional pipeline-specific keyword arguments (like `max_length`, `top_k`, etc.)    |


more details here : https://huggingface.co/docs/transformers/pipeline_tutorial

### Hugging Face Tasks

more details here : https://huggingface.co/tasks

In [ ]:
from transformers import pipeline
#---------------------------------------------------#
#                     NLP TASKS                     #
#---------------------------------------------------#

'''
1. Text Classification: Assigning a category to a piece of text.
Sentiment Analysis
Topic Classification
Spam Detection '''

classifier = pipeline("text-classification")

'''
2. Token Classification: Assigning labels to individual tokens in a sequence.
Named Entity Recognition (NER)
Part-of-Speech Tagging
'''

token_classifier = pipeline("token-classification")

'''
3. Question Answering: Extracting an answer from a given context based on a question.
'''
question_answerer = pipeline("question-answering")

'''
4. Text Generation: Generating text based on a given prompt.
Language Modeling
Story Generation

'''

text_generator = pipeline("text-generation")

'''
5. Summarization: Condensing long documents into shorter summaries.
'''

summarizer = pipeline("summarization")

'''
Translation: Translating text from one language to another.
'''

translator = pipeline("translation",
                      model="Helsinki-NLP/opus-mt-en-fr")

'''
6. Text2Text Generation: General-purpose text transformation, including summarization and translation.
'''

text2text_generator = pipeline("text2text-generation")

'''
7. Fill-Mask: Predicting the masked token in a sequence.
'''

fill_mask = pipeline("fill-mask")

'''
8. Feature Extraction: Extracting hidden states or features from text.
'''

feature_extractor = pipeline("feature-extraction")

'''
9. Sentence Similarity: Measuring the similarity between two sentences.
'''
sentence_similarity = pipeline("sentence-similarity")

#---------------------------------------------------#
#             Computer Vision TASKS                 #
#---------------------------------------------------#

'''
1. Image Classification: Classifying the main content of an image.

'''

image_classifier = pipeline("image-classification")

'''
2. Object Detection: Identifying objects within an image and their bounding boxes.
'''

object_detector = pipeline("object-detection")

'''
3. Image Segmentation: Segmenting different parts of an image into classes.
'''

image_segmenter = pipeline("image-segmentation")

'''
4. Image Generation: Generating images from textual descriptions (using DALL-E or similar models).
'''

#---------------------------------------------------#
#             Speech Processing TASKS               #
#---------------------------------------------------#

'''
1. utomatic Speech Recognition (ASR): Converting spoken language into text.
'''

speech_recognizer = pipeline("automatic-speech-recognition")

'''
2. Speech Translation: Translating spoken language from one language to another.
3. Audio Classification: Classifying audio signals into predefined categories.
'''

#---------------------------------------------------#
#                   Multimodal TASKS                #
#---------------------------------------------------#

'''
1. Image Captioning: Generating a textual description of an image.
'''
image_captioner = pipeline("image-to-text")
'''
2. Visual Question Answering (VQA): Answering questions about the content of an image.
'''

#---------------------------------------------------#
#                     Other TASKS                   #
#---------------------------------------------------#
'''
1. Table Question Answering: Answering questions based on tabular data.
'''
table_qa = pipeline("table-question-answering")

'''
2. Document Question Answering: Extracting answers from documents like PDFs.

'''
doc_qa = pipeline("document-question-answering")
'''
3. Time Series Forecasting: Predicting future values in time series data (not directly supported in the main Transformers library but available through extensions).
'''

# NLp Task

#### Sentiment Analysis

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
result = classifier("I was so not happy with the last Mission Impossible Movie")
print(result)


In [ ]:
pipeline(task = "sentiment-analysis")("I was confused with the Barbie Movie")

In [ ]:
pipeline(task = "sentiment-analysis")\
                                      ("Everyday lots of LLMs papers are published about LLMs Evlauation. \
                                      Lots of them Looks very Promising. \
                                      I am not sure if we CAN actually Evaluate LLMs. \
                                      There is still lots to do.\
                                      Don't you think?")

In [ ]:
pipeline(task = "sentiment-analysis", model="facebook/bart-large-mnli")\
                                      ("Everyday lots of LLMs papers are published about LLMs Evlauation. \
                                      Lots of them Looks very Promising. \
                                      I am not sure if we CAN actually Evaluate LLMs. \
                                      There is still lots to do.\
                                      Don't you think?")


### Batch Senteniment Analysis

In [ ]:
classifier = pipeline(task = "sentiment-analysis")

task_list = ["I really like Autoencoders, best models for Anomaly Detection", \
            "I am not sure if we CAN actually Evaluate LLMs.", \
            "PassiveAgressive is the name of a Linear Regression Model that so many people do not know.",\
            "I hate long Meetings."]
classifier(task_list)

In [ ]:
classifier = pipeline(task = "sentiment-analysis", model = "SamLowe/roberta-base-go_emotions")

task_list = ["I really like Autoencoders, best models for Anomaly Detection", \
            "I am not sure if we CAN actually Evaluate LLMs.", \
            "PassiveAgressive is the name of a Linear Regression Model that so many people do not know. It is pretty funny name for a Regression Model.",\
            "I hate long Meetings."]
classifier(task_list)

### Text Generation

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

text_generator = pipeline("text-generation", model="distilbert/distilgpt2")
generated_text = text_generator("Today is a rainy day in London",
                                truncation=True,
                                num_return_sequences = 2)
print("Generated_text:\n ", generated_text[0]['generated_text'])

### Question Answering

In [ ]:
from transformers import pipeline

qa_model = pipeline("question-answering")
question = "What is my job?"
context = "I am developing AI models with Python."
qa_model(question = question, context = context)

# Evalution Matrixs

In [ ]:
!pip install evaluate
!pip install rouge_score
!pip install evaluate[visualization]
!pip install -U datasets fsspec

In [ ]:
import evaluate
accuracy = evaluate.load("accuracy")


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

In [ ]:

# Load dataset subsets
train_dataset = load_dataset("imdb", split="train[:100]")
test_dataset = load_dataset("imdb", split="test[:100]")

# Load pretrained model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize and format
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

# Tokenize train and test
train_encoded = train_dataset.map(tokenize, batched=True)
test_encoded = test_dataset.map(tokenize, batched=True)

# Format for PyTorch
train_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Define Trainer with metrics
args = TrainingArguments(output_dir="eval-check", per_device_eval_batch_size=32,report_to="none")

trainer = Trainer(model=model, args=args, compute_metrics=compute_metrics)

metrics = trainer.evaluate(eval_dataset=test_encoded)

print(metrics)

# Fine Tunning IMDB

## Step 1: Install Necessary Libraries

In [ ]:
!pip install datasets

## Step 2: Load and Prepare the Dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset('imdb')

In [ ]:
dataset

In [ ]:
dataset["train"][0]

## Step 3: Preprocess the Data
Tokenize the dataset using the tokenizer associated with the pre-trained model.

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)


In [ ]:
tokenized_datasets

In [ ]:
tokenized_datasets["train"][0]

## Step 4: Set Up the Training Arguments
Specify the hyperparameters and training settings.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',          # Output directory
    eval_strategy ="epoch",     # Evaluate every epoch
    learning_rate=2e-5,              # Learning rate
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=16,   # Batch size for evaluation
    num_train_epochs=1,              # Number of training epochs
    weight_decay=0.01,               # Strength of weight decay
)
training_args

## Step 5: Initialize the Model
Load the pre-trained model and define the training procedure.

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer

# Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test']
)


## Step 6: Train the Model
Fine-tune the pre-trained model on your specific dataset.

In [ ]:
# Train the model
trainer.train()

## Step 8: Save the Fine-Tuned Model
Save the fine-tuned model for later use.

In [ ]:
# Save the model
model.save_pretrained('./fine-tuned-model')
tokenizer.save_pretrained('./fine-tuned-tokenizer')


## Text-to-Image Generation using Stable Diffusion and Diffusers



In [ ]:
#diffusers is a hugging face page for using diffusion models from huggingface hub
!pip install diffusers transformers accelerate

In [ ]:
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt
import torch

In [ ]:
model_id1 = "dreamlike-art/dreamlike-diffusion-1.0"
model_id2 = "stabilityai/stable-diffusion-xl-base-1.0"

pipe = StableDiffusionPipeline.from_pretrained(model_id1, torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to("cuda")

In [ ]:
prompt = """dreamlikeart, a grungy woman with rainbow hair, travelling between dimensions, dynamic pose, happy, soft eyes and narrow chin,
extreme bokeh, dainty figure, long hair straight down, torn kawaii shirt and baggy jeans
"""

In [ ]:
image = pipe(prompt).images[0]

In [ ]:
print("[PROMPT]: ",prompt)
plt.imshow(image);
plt.axis('off');

In [ ]:
prompt2 = """A girl is sittig on a chair & She is accompanied by her tiger. Make sure to keep it cinematic and color to be golden iris
"""

image = pipe(prompt2).images[0]
print('[PROMPT]: ',prompt2)
plt.imshow(image);
plt.axis('off');

### Working with Stable Diffusion parameters

* Negative prompting
* num_inference_steps
* height
* weight
* num_images_per_prompt


more details here : https://huggingface.co/docs/diffusers/using-diffusers/loading

In [ ]:
def generate_image(pipe, prompt, params):
  img = pipe(prompt, **params).images

  num_images = len(img)
  if num_images>1:
    fig, ax = plt.subplots(nrows=1, ncols=num_images)
    for i in range(num_images):
      ax[i].imshow(img[i]);
      ax[i].axis('off');

  else:
    fig = plt.figure()
    plt.imshow(img[0]);
    plt.axis('off');
  plt.tight_layout()

In [ ]:
prompt = "dreamlike, beautiful girl playing the festival of colors, draped in traditional Indian attire, throwing colors"

params = {}
generate_image(pipe, prompt, params)
#num inference steps
params = {'num_inference_steps': 100}

generate_image(pipe, prompt, params)

In [ ]:
#height width
params = {'num_inference_steps': 100, 'width': 512, 'height': int(1.5*640)}

generate_image(pipe, prompt, params)

In [ ]:
#num_images_per_prompt
params = {'num_inference_steps': 100, 'num_images_per_prompt': 2}

generate_image(pipe, prompt, params)

In [ ]:
# negative_prompt
params = {'num_inference_steps': 100, 'num_images_per_prompt': 2, 'negative_prompt': 'ugly, distorted, low quality'}

generate_image(pipe, prompt, params)

### Text_to_speech_generation_with_LLM

In [ ]:
from transformers import pipeline
text = "Python is a high-level, general-purpose programming language."

pipe = pipeline("text-to-speech", model="suno/bark-small", device="cuda")

output = pipe(text)
output

In [ ]:
from IPython.display import Audio

Audio(output["audio"], rate=output["sampling_rate"])